# Multilingual Image Captioning: train on a Colab GPU

Runtime > Change runtime type > **T4 GPU**. This notebook clones the repo, prepares an MS-COCO 2014 subset, caches CNN features, trains InceptionV3 and ResNet50 runs at the same settings, evaluates them and zips the checkpoint for a GitHub Release.

Rough timings on a T4 with 6,000 images: download ~5 min, features ~3 min per backbone, training ~1-2 min per epoch.

In [ ]:
!nvidia-smi -L
!git clone https://github.com/Anudeepthi905/multilingual-image-captioning.git
%cd multilingual-image-captioning
# Keep Colab's own TensorFlow/NumPy (built for its CUDA); install everything else.
!grep -vE '^(tensorflow|keras|numpy)==' requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt pytest
import tensorflow as tf; print(tf.__version__, tf.config.list_physical_devices('GPU'))

In [ ]:
# Settings: raise NUM_IMAGES (e.g. 30000) if you have time; add --full-zip for > ~20k images.
NUM_IMAGES = 6000
VOCAB = 5000
EPOCHS = 20
!python -m pytest -q
!python -m src.data --num-images {NUM_IMAGES}

In [ ]:
# Cache CNN features once per backbone (training never re-runs the CNN).
!python -m src.features --backbone inceptionv3 --num-images {NUM_IMAGES} --batch-size 64
!python -m src.features --backbone resnet50 --num-images {NUM_IMAGES} --batch-size 64

In [ ]:
!python -m src.train --backbone inceptionv3 --vocab-size {VOCAB} --num-images {NUM_IMAGES} --epochs {EPOCHS}
!python -m src.train --backbone resnet50 --vocab-size {VOCAB} --num-images {NUM_IMAGES} --epochs {EPOCHS}
# Optional: larger vocabulary
# !python -m src.train --backbone inceptionv3 --vocab-size 7000 --num-images {NUM_IMAGES} --epochs {EPOCHS}

In [ ]:
INC = f'inceptionv3_v{VOCAB}_n{NUM_IMAGES}_e{EPOCHS}'
RES = f'resnet50_v{VOCAB}_n{NUM_IMAGES}_e{EPOCHS}'
for run in [INC, RES]:
    !python -m src.evaluate --run {run}
    !python -m src.evaluate --run {run} --beam 3
import pandas as pd; pd.read_csv('results/metrics.csv')

In [ ]:
from IPython.display import Image, display
!python -m src.caption --image assets/samples/$(ls assets/samples | grep -m1 jpg) --run {INC} --beam 3 --plot
for f in ['results/comparison.png', f'results/loss_{INC}.png', f'results/loss_{RES}.png']:
    display(Image(f))

In [ ]:
# Package the best run for a GitHub Release (attach checkpoint.zip to the v1.0 release)
# and download the results folder to commit metrics.csv and the plots.
!cd checkpoints && zip -r ../checkpoint.zip {INC}
!zip -r results.zip results README.md
from google.colab import files
files.download('checkpoint.zip'); files.download('results.zip')